## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys
sys.setrecursionlimit(20000)

def main():
    # 快速读取所有输入数据
    input_data = list(map(int, sys.stdin.buffer.read().split()))
    if not input_data:
        return
    
    data_idx = 0
    total_elements = input_data[data_idx]; data_idx += 1
    magic_x = input_data[data_idx]; data_idx += 1
    magic_y = input_data[data_idx]; data_idx += 1
    nums = input_data[data_idx : data_idx + total_elements]; data_idx += total_elements
    pos_map = [0] * total_elements
    for idx, val in enumerate(nums):
        pos_map[val] = idx
        
    operations_log = []
    
    def synchronize_positions():
        for i in range(total_elements):
            pos_map[nums[i]] = i
            
    def apply_magic_swap():
        operations_log.append(0)
        for i in range(total_elements):
            if nums[i] == magic_x:
                nums[i] = magic_y
            elif nums[i] == magic_y:
                nums[i] = magic_x
        synchronize_positions()
        
    def apply_shift(shift_amount):
        if shift_amount == 0:
            return
        operations_log.append(shift_amount)
        for i in range(total_elements):
            nums[i] = (nums[i] + shift_amount) % total_elements
        synchronize_positions()
        
    def apply_xor(xor_mask):
        if xor_mask == 0:
            return
        operations_log.append(-xor_mask)
        for i in range(total_elements):
            nums[i] ^= xor_mask
        synchronize_positions()
    diff = (magic_x - magic_y) % total_elements
    bit_step = total_elements if diff == 0 else (diff & -diff)
    offset_cache = {}
    
    def calculate_offsets(target_x, target_y):
        cache_key = (target_x, target_y)
        if cache_key in offset_cache:
            return offset_cache[cache_key]
            
        delta = (target_y - target_x + total_elements - bit_step + total_elements) % total_elements
        offset_a = 0
        offset_b = 0
        step_size = total_elements // 2
        
        while step_size >= 2 * bit_step:
            if delta >= step_size:
                delta -= step_size
                offset_b += step_size // 2
            else:
                offset_a += step_size // 2
            step_size //= 2
            
        offset_a += total_elements // 2
        offset_a += (target_x & (bit_step - 1))
        offset_b += (target_x & (bit_step - 1))
        
        offset_cache[cache_key] = (offset_a, offset_b)
        return offset_a, offset_b
        
    def resolve_swap(elem_c, elem_d):
        if (elem_c // bit_step) % 2 == (elem_d // bit_step) % 2:
            if (elem_c // bit_step) % 2 == 0:
                pivot = (elem_c & (bit_step - 1)) + bit_step
            else:
                pivot = (elem_c & (bit_step - 1))
            resolve_swap(elem_c, pivot)
            resolve_swap(elem_d, pivot)
            resolve_swap(elem_c, pivot)
        else:
            pa, pb = calculate_offsets(magic_x, magic_y)
            pc, pd = calculate_offsets(elem_c, elem_d)
            
            apply_shift((pc - elem_c + total_elements) % total_elements)
            apply_xor(pc ^ pa)
            apply_shift((magic_x - pa + total_elements) % total_elements)
            apply_magic_swap()
            apply_shift((pa - magic_x + total_elements) % total_elements)
            apply_xor(pc ^ pa)
            apply_shift((elem_c - pc + total_elements) % total_elements)

    # 分治策略类
    class DivideAndConquerPerm:
        __slots__ = ('elements', 'size', 'ops_history')
        def __init__(self, arr, sz):
            self.elements = arr[:]
            self.size = sz
            self.ops_history = []
            
        def execute_analysis(self):
            if self.size == 0:
                return True
            
            visited = [False] * self.size
            for val in self.elements:
                if val >= self.size:
                    return False
                visited[val] = True
            if not all(visited):
                return False
                
            if self.size == 1:
                return True
                
            half_size = self.size // 2
            left_sub = [0] * half_size
            right_sub = [0] * half_size
            
            curr_elements = self.elements
            for i in range(half_size):
                left_sub[i] = curr_elements[i * 2] // 2
                right_sub[i] = curr_elements[i * 2 + 1] // 2
                
            left_solver = DivideAndConquerPerm(left_sub, half_size)
            right_solver = DivideAndConquerPerm(right_sub, half_size)
            
            if not left_solver.execute_analysis() or not right_solver.execute_analysis():
                return False
                
            history = self.ops_history
            if curr_elements[0] & 1:
                history.append(1 if self.size == 2 else -1)
                
            checksum_left = 0
            for val in left_solver.ops_history:
                if val > 0:
                    history.append(-1)
                    history.append(1)
                else:
                    history.append(val * 2)
                    checksum_left ^= -val * 2
                    
            if checksum_left:
                history.append(-checksum_left)
                
            checksum_right = 0
            for val in right_solver.ops_history:
                if val > 0:
                    history.append(1)
                    history.append(-1)
                else:
                    history.append(val * 2)
                    checksum_right ^= -val * 2
                    
            if (checksum_right & half_size) != (checksum_left & half_size):
                for _ in range(self.size // 4):
                    history.append(-1)
                    history.append(1)
                    
            if checksum_left >= half_size:
                checksum_left -= half_size
            if checksum_right >= half_size:
                checksum_right -= half_size
                
            if checksum_left != checksum_right:
                return False
                
            # 合并连续的同类操作
            compressed_history = []
            for val in history:
                if not compressed_history:
                    compressed_history.append(val)
                elif val < 0 and compressed_history[-1] < 0:
                    compressed_history[-1] = -((-compressed_history[-1]) ^ (-val))
                    if compressed_history[-1] == 0:
                        compressed_history.pop()
                else:
                    compressed_history.append(val)
                    
            self.ops_history = compressed_history
            return True
    if bit_step > 1:
        initial_mask = [nums[i] & (bit_step - 1) for i in range(total_elements)]
        initial_solver = DivideAndConquerPerm(initial_mask, bit_step)
        if not initial_solver.execute_analysis():
            print(-1)
            return
            
        for val in initial_solver.ops_history:
            if val > 0:
                apply_shift(val)
            else:
                apply_xor(-val)
    for rem in range(bit_step):
        bucket = []
        for j in range(rem, total_elements, bit_step):
            bucket.append(nums[j])
        bucket.sort()
        
        bucket_ptr = 0
        is_valid = True
        for j in range(rem, total_elements, bit_step):
            if bucket[bucket_ptr] != j:
                is_valid = False
                break
            bucket_ptr += 1
            
        if not is_valid:
            print(-1)
            return
            
        for j in range(rem, total_elements, bit_step):
            if nums[j] != j:
                resolve_swap(j, nums[j])
                
    # 最终完整性校验
    for i in range(total_elements):
        if nums[i] != i:
            print(-1)
            return
            
    # 标准化格式并输出结果
    print(len(operations_log))
    output_buffer = []
    for op in operations_log:
        if op == 0:
            output_buffer.append("0")
        elif op > 0:
            output_buffer.append(f"2 {op}")
        else:
            output_buffer.append(f"1 {-op}")
            
    sys.stdout.write("\n".join(output_buffer))

if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys
def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    ptr = 0
    n_tokens = len(input_data)
    while ptr < n_tokens:
        N = int(input_data[ptr])
        L = int(input_data[ptr+1])
        Maxn = int(input_data[ptr+2])
        S = int(input_data[ptr+3])
        ptr += 4
        nodes = []
        for _ in range(N):
            p = int(input_data[ptr])
            c = int(input_data[ptr+1])
            ptr += 2
            if p <= L:
                nodes.append((p, c))
        nodes.append((L, 0))
        nodes.sort(key=lambda x: x[0])
        dp = [-1] * (S + 1)
        dp[0] = Maxn
        
        last_p = 0
        
        for curr_p, curr_c in nodes:
            dist = curr_p - last_p
            if dist > 0:
                for j in range(S + 1):
                    if dp[j] != -1:
                        dp[j] -= dist
                        if dp[j] < 0:
                            dp[j] = -1
            if curr_c <= S:
                for j in range(S, curr_c - 1, -1):
                    if dp[j - curr_c] >= 0:
                        if Maxn > dp[j]:
                            dp[j] = Maxn
            
            last_p = curr_p
        success = False
        for j in range(S + 1):
            if dp[j] >= 0:
                success = True
        if success:
            print("Yes")
        else:
            print("No")

if __name__ == '__main__':
    solve()

## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;
vector<int> radA, radB;

void buildManacher(string &str, vector<int> &radius)
{
    string tmp = "@";

    int len = str.size();

    for (int i = 0; i < len; i++)
    {
        tmp += '*';
        tmp += str[i];
    }

    tmp += '*';

    int m = tmp.size();

    radius.resize(m + 1, 0);

    int rightMost = 0;
    int mid = 0;

    for (int i = 1; i < m; i++)
    {
        if (i < rightMost)
            radius[i] = min(rightMost - i, radius[2 * mid - i]);
        else
            radius[i] = 1;

        while (
            i - radius[i] >= 0 &&
            i + radius[i] < m &&
            tmp[i - radius[i]] == tmp[i + radius[i]]
        )
        {
            radius[i]++;
        }

        if (i + radius[i] > rightMost)
        {
            rightMost = i + radius[i];
            mid = i;
        }
    }

    str = tmp;
}

int main()
{
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    string A, B;

    cin >> A >> B;

    buildManacher(A, radA);
    buildManacher(B, radB);

    int answer = 0;

    for (int pos = 1; pos < 2 * n + 2; pos++)
    {
        int cur = max(radA[pos], radB[pos - 2]);

        while (A[pos - cur] == B[pos - 2 + cur])
        {
            cur++;
        }

        answer = max(answer, cur);
    }

    cout << answer - 1 << '\n';

    return 0;
}

## D 优惠券

In [ ]:
#include <iostream>
#include <string.h>
#include <vector>
#include <set>
#include <algorithm>
#include <cstdio>
#include <map>
using namespace std;
char op_history[555555];       // 记录每一行的操作字符（P, U, ?）
int coupon_history_id[555555]; // 记录每一行操作的优惠券ID
int main() {
    int total_ops;
    while (~scanf("%d", &total_ops)) {
        int first_error_line = -1;
        set<int> wildcard_positions; // 存放所有问号 '?' 的行号
        int is_legal = 1;            // 逻辑合法标记
        map<int, int> last_seen_line; 
        
        char op_str[2];
        for (int i = 1; i <= total_ops; i++) {
            scanf("%s", op_str);
            op_history[i] = op_str[0];
            
            if (op_history[i] != '?') {
                scanf("%d", &coupon_history_id[i]);
            }
            if (is_legal == 0) {
                continue;
            }
            if (op_history[i] == '?') {
                wildcard_positions.insert(i);
                continue;
            }
            
            int current_coupon = coupon_history_id[i];
            if (last_seen_line[current_coupon] != 0) {
                int prev_line = last_seen_line[current_coupon];
                if (op_history[prev_line] == op_history[i]) {
                    // 必须在两次相同操作之间找一个问号来缓冲
                    set<int>::iterator it = wildcard_positions.lower_bound(prev_line);
                    if (it == wildcard_positions.end()) {
                        first_error_line = i;
                        is_legal = 0;
                    } else {
                        wildcard_positions.erase(it); // 成功用问号救活
                    }
                }
                last_seen_line[current_coupon] = i;
            } 
            else {
                // 冲突情况2：第一次出现就是 'U' (使用)，说明没买就用，非法！
                if (op_history[i] == 'U' || op_history[i] == 'O') { // 兼容旧字母 O 和新字母 U
                    // 必须在当前行之前找一个问号假装成 'P' (购买)
                    set<int>::iterator it = wildcard_positions.lower_bound(0);
                    if (it == wildcard_positions.end()) {
                        first_error_line = i;
                        is_legal = 0;
                    } else {
                        wildcard_positions.erase(it); // 成功用之前的问号救活
                    }
                }
                last_seen_line[current_coupon] = i;
            }
        }
        if (is_legal && !wildcard_positions.empty()) {
            first_error_line = *wildcard_positions.begin();
        }   
        printf("%d\n", first_error_line);
    }
    return 0;
}

## E 任意点

In [ ]:
import sys
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.count = n
    def find(self, i):
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def unite(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j
            self.count -= 1
def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    n = int(input_data[0])
    points = []
    idx = 1
    for _ in range(n):
        x = int(input_data[idx])
        y = int(input_data[idx+1])
        points.append((x, y))
        idx += 2
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i + 1, n):
            if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                uf.unite(i, j)

    print(uf.count - 1)

if __name__ == '__main__':
    solve()

## F 通配符匹配

In [ ]:
#include<cmath>
#include<cstdio>
#include<vector>
#include<cstring>
#include<iostream>
#include<algorithm>
#define INF_VAL 0x7f7f7f7f
#define for_vec(it,x) for (vector<int>::iterator it=x.begin();it!=x.end();it++)
using namespace std;
typedef long long ll;
typedef unsigned int ui;
typedef unsigned long long ull;

inline char gc(){
    static char buf[1000000],*p1=buf,*p2=buf;
    return p1==p2&&(p2=(p1=buf)+fread(buf,1,1000000,stdin),p1==p2)?EOF:*p1++;
}
inline int frd(){
    int x=0,f=1;char ch=gc();
    for (;ch<'0'||ch>'9';ch=gc())   if (ch=='-')    f=-1;
    for (;ch>='0'&&ch<='9';ch=gc()) x=(x<<1)+(x<<3)+ch-'0';
    return x*f;
}
inline int read(){
    int x=0,f=1;char ch=getchar();
    for (;ch<'0'||ch>'9';ch=getchar())  if (ch=='-')    f=-1;
    for (;ch>='0'&&ch<='9';ch=getchar())    x=(x<<1)+(x<<3)+ch-'0';
    return x*f;
}
inline void print(int x){
    if (x<0)    putchar('-'),x=-x;
    if (x>9)    print(x/10);
    putchar(x%10+'0');
}

const int LIMIT=1e5;
int match_positions[LIMIT+10], match_cnt, pattern_len;
bool HAS_HEAD_STAR, HAS_TAIL_STAR;

struct Automaton {
    int next_node[LIMIT+10][26], fail_link[LIMIT+10];
    int node_cnt, rt, match_num, segment_len;
    vector<int> match_ends[LIMIT+10];

    void insert(char *src, int l, int r){
        static char temp_str[LIMIT+10];
        for (int i=l; i<r; i++) temp_str[i-l] = src[i];
        int len = r-l, p = rt; 
        temp_str[len++] = '?';

        for (int i=0; i<len; i++){
            while (temp_str[i]=='?' && i<len){
                if (i && temp_str[i-1]!='?') {
                    match_ends[p].push_back(i-1);
                    match_num++;
                }
                p = rt, i++;
            }
            if (i>=len) break;
            if (!next_node[p][temp_str[i]-'a']) next_node[p][temp_str[i]-'a'] = ++node_cnt;
            p = next_node[p][temp_str[i]-'a'];
        }
        segment_len = --len;
    }

    void make_fail(){
        static int queue_buf[LIMIT+10];
        int head=1, tail=0;
        for (int i=0; i<26; i++) if (next_node[rt][i]) queue_buf[++tail] = next_node[rt][i];
        
        for (; head<=tail; head++){
            int current_node = queue_buf[head];
            for_vec(it, match_ends[fail_link[current_node]]) match_ends[current_node].push_back(*it);
            for (int i=0; i<26; i++){
                if (next_node[current_node][i]){
                    int son_node = next_node[current_node][i];
                    fail_link[son_node] = next_node[fail_link[current_node]][i];
                    queue_buf[++tail] = son_node;
                } else {
                    next_node[current_node][i] = next_node[fail_link[current_node]][i];
                }
            }
        }
    }

    bool check(char *src, bool is_first, bool is_last){
        static int match_scores[LIMIT+10];
        memset(match_scores, 0, sizeof(match_scores));
        int len = strlen(src), p = rt;

        for (int i=0; i<len; i++){
            p = next_node[p][src[i]-'a'];
            for_vec(it, match_ends[p]) if (i>=*it) match_scores[i-*it]++;
        }

        for (int i=0; i<len; i++){
            if (match_scores[i] != match_num) continue;
            if (!match_cnt){
                if (!HAS_HEAD_STAR && is_first && i) break;
                if (!HAS_TAIL_STAR && is_last && len-i!=segment_len) break;
                match_positions[++match_cnt] = i + segment_len;
                return 1;
            } else {
                if (i < match_positions[match_cnt]) continue;
                if (!HAS_TAIL_STAR && is_last && len-i!=segment_len) break;
                match_positions[++match_cnt] = i + segment_len;
                return 1;
            }
        }
        return 0;
    }
} AC_Automaton[10];

char pattern[LIMIT+10];
char queryStr[LIMIT+10];

int main(){
    // 这里读入的是主模式串，如 *abc?de*
    if(scanf("%s", pattern) == EOF) return 0;
    int len = strlen(pattern), total_segments = 0; 
    pattern_len = len;

    HAS_HEAD_STAR = (pattern[0] == '*');
    HAS_TAIL_STAR = (pattern[len-1] == '*');
    pattern[len++] = '*';

    for (int i=0, last_star_pos=0; i<len; i++){
        if (pattern[i] == '*'){
            if (i > last_star_pos) AC_Automaton[total_segments++].insert(pattern, last_star_pos, i);
            last_star_pos = i + 1;
        }
    }

    for (int i=0; i<total_segments; i++) AC_Automaton[i].make_fail();
    for (int Q=read(); Q; Q--){
        if (!pattern_len){
            printf("NO\n");
            continue;
        }
        if (!total_segments){
            printf("YES\n");
            continue;
        }
        memset(queryStr, 0, sizeof(queryStr));
        memset(match_positions, 0, sizeof(match_positions)); 
        match_cnt = 0;

        scanf("%s", queryStr);
        bool is_matched = 1;

        for (int i=0; i<total_segments; i++){
            if (!AC_Automaton[i].check(queryStr, i==0, i==total_segments-1)){
                is_matched = 0;
                break;
            }
        }
        printf(is_matched ? "YES\n" : "NO\n");
    }
    return 0;
}

## G 汉诺塔

In [ ]:
import sys
def solve():
    # 快速读取所有输入数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    total_plates = int(input_data[0])
    rules = input_data[1:7] 
    min_steps = [[0] * 305 for _ in range(4)]
    target_peg = [[0] * 305 for _ in range(4)]
    is_rule_set = [False] * 4
    for rule in rules:
        from_peg = ord(rule[0]) - ord('A') + 1
        to_peg = ord(rule[1]) - ord('A') + 1
        if is_rule_set[from_peg]:
            continue   
        min_steps[from_peg][1] = 1
        target_peg[from_peg][1] = to_peg
        is_rule_set[from_peg] = True
    # 2. 动态规划递推：处理 2 到 total_plates 个盘子
    for i in range(2, total_plates + 1):
        for from_peg in range(1, 4):
            mid_peg = target_peg[from_peg][i - 1]
            other_peg = 6 - from_peg - mid_peg
            if target_peg[mid_peg][i - 1] == other_peg:
                target_peg[from_peg][i] = other_peg
                min_steps[from_peg][i] = min_steps[from_peg][i - 1] + 1 + min_steps[mid_peg][i - 1]
            elif target_peg[mid_peg][i - 1] == from_peg:
                target_peg[from_peg][i] = mid_peg
                min_steps[from_peg][i] = (min_steps[from_peg][i - 1] + 1 + 
                                          min_steps[mid_peg][i - 1] + 1 + 
                                          min_steps[from_peg][i - 1])

    # 输出从 1 号柱子（A柱）移走全部 n 个盘子的总步数
    print(min_steps[1][total_plates])

if __name__ == '__main__':
    solve()

## H 马步距离

In [ ]:
import sys
def solve():
    # 一次性读取所有输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    xp, yp, xs, ys = map(int, input_data)
    x = abs(xp - xs)
    y = abs(yp - ys)
    if x < y:
        x, y = y, x
    if x == 1 and y == 0:
        print(3)
        return
    if x == 2 and y == 2:
        print(4)
        return
    ans = max((x + 1) // 2, (x + y + 2) // 3)
    if (ans % 2) != ((x + y) % 2):
        ans += 1
        
    print(ans)

if __name__ == '__main__':
    solve()

## I 直方图最大矩形

In [ ]:
class Solution:
    def largestRectangleArea(self, heights: list[int]) -> int:
        if not heights:
            return 0
        heights.append(0)
        stack = [-1] 
        max_area = 0
        
        for i in range(len(heights)):
            while len(stack) > 1 and heights[i] < heights[stack[-1]]:
                h = heights[stack.pop()] 
                w = i - stack[-1] - 1  
                max_area = max(max_area, h * w)
                
            stack.append(i)
            
        return max_area

## J 消防局的设立

In [ ]:
import sys
from collections import deque

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    fa = [0] * (n + 1)
    depth = [0] * (n + 1)
    adj = [[] for _ in range(n + 1)]
    fa[1] = 1
    depth[1] = 1
    for i in range(2, n + 1):
        parent = int(input_data[i - 1])
        fa[i] = parent
        depth[i] = depth[parent] + 1
        adj[i].append(parent)
        adj[parent].append(i)
    nodes_sorted_by_depth = sorted(range(1, n + 1), key=lambda x: depth[x], reverse=True)
    
    covered = [False] * (n + 1)
    ans = 0
    def bfs_cover(start_node):
        queue = deque([(start_node, 0)]) 
        covered[start_node] = True
        
        while queue:
            u, dist = queue.popleft()
            
            if dist == 2:
                continue
                
            for v in adj[u]:
                covered[v] = True
                if dist < 2:
                    queue.append((v, dist + 1))

    # 开始贪心放置消防局
    for u in nodes_sorted_by_depth:
        if not covered[u]:
            grandfather = fa[fa[u]]
            if grandfather == 0:
                grandfather = 1
            ans += 1
            bfs_cover(grandfather)
            
    print(ans)

if __name__ == '__main__':
    solve()